# Демонстрация вызова Loginom REST API + Mistral

Этот ноутбук показывает полный цикл:
1. POST-запрос к `GetUserHistory` в Loginom
2. Парсинг ответа в DataFrame
3. Формирование промпта
4. Получение персональной рекомендации от Mistral

In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from mistralai import Mistral

load_dotenv()  # читаем .env из папки проекта

MISTRAL_API_KEY = os.getenv('MISTRAL_API_KEY')
LOGINOM_URL     = os.getenv('LOGINOM_URL')

assert MISTRAL_API_KEY, 'Задайте MISTRAL_API_KEY в файле .env'
assert LOGINOM_URL,     'Задайте LOGINOM_URL в файле .env'

print('Конфигурация загружена.')
print(f'LOGINOM_URL = {LOGINOM_URL}')

## 1. Вызов GetUserHistory

In [ ]:
USER_ID = 1  # <-- поменяйте на нужный user_id

response = requests.post(
    f'{LOGINOM_URL}/GetUserHistory',
    json={'user_id': USER_ID},
    timeout=15,
)
response.raise_for_status()

data = response.json()
print('Статус ответа:', response.status_code)
print('Ключи JSON:', list(data.keys()))

## 2. Парсинг в DataFrame

In [ ]:
rows = data.get('DataSet', {}).get('Rows', [])
df = pd.DataFrame(rows)
print(f'Получено строк: {len(df)}')
df.head(10)

## 3. Формирование текста для промпта

In [ ]:
lines = []
for i, row in enumerate(rows[:10], start=1):
    name       = row.get('product_name', '—')
    department = row.get('department', '—')
    aisle      = row.get('aisle', '—')
    orders     = row.get('order_count', row.get('orders_count', '?'))
    lines.append(f'{i}. {name} (отдел: {department}, категория: {aisle}, заказов: {orders})')

products_text = '\n'.join(lines)
print(products_text)

## 4. Запрос к Mistral

In [ ]:
client = Mistral(api_key=MISTRAL_API_KEY)

system_prompt = (
    'Вы — персональный помощник покупателя интернет-магазина продуктов. '
    'Ваша задача — анализировать историю покупок клиента и давать дружелюбные, '
    'полезные рекомендации. Отвечайте на русском языке, обращайтесь к покупателю '
    'на «вы». Ответ должен занимать 5–6 предложений: сначала кратко охарактеризуйте '
    'предпочтения клиента, затем дайте 1–2 конкретных совета по новым товарам или '
    'категориям, которые могут ему понравиться. Тон — тёплый и позитивный.'
)

user_message = (
    f'Покупатель #{USER_ID} чаще всего заказывает следующие товары:\n\n'
    f'{products_text}\n\n'
    'Пожалуйста, дайте персональную рекомендацию этому покупателю.'
)

resp = client.chat.complete(
    model='mistral-small-latest',
    messages=[
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_message},
    ],
)

recommendation = resp.choices[0].message.content.strip()
print('=== Персональная рекомендация ===')
print(recommendation)